***Resume Assistant Chatbot that clarifies any queries from that resume using RAG pipeline***

***
1)Data Extraction
<br>2)Chunking using Topic Based Chunking 
<br>3)Embedding using Hugging Face 
<br>4)pineCone Vector DB
<br>5)Similarity search using cosine similarity
<br>6)Fetching into LLM
<br>7)Prompt creation
<br>8)streamlit for deployment
***

**1) Convert resume into chunks**

In [56]:

import pdfplumber
from pydantic import BaseModel, Field, ValidationError
import re

# Define a Pydantic model for structured resume chunks
class ResumeChunk(BaseModel):
    section: str = Field(..., description="Section of the resume (e.g., Experience, Education)")
    text: str = Field(..., min_length=10, description="Chunk of resume text")
    chunk_id: str

# Function to extract text from a PDF file using pd
# fplumber
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            extracted_text = page.extract_text()
            if extracted_text:
                text += extracted_text + "\n"
    return text.strip()

# Function to split text into labeled chunks based on common resume section titles
def chunk_resume_with_labels(resume_text):
    # Define section headers (with case insensitivity)
    section_headers = ["SUMMARY", "EXPERIENCE", "EDUCATION", "SKILLS & TOOLS", "CERTIFICATIONS", "PROJECTS"]

    # Create regex pattern to detect section headers (allowing extra spaces and newlines)
    section_pattern = re.compile(rf"(?i)^\s*({'|'.join(section_headers)})\s*$", re.MULTILINE)
    print(section_pattern)
    print('------------------------')

    # Split text based on detected section headers
    sections = section_pattern.split(resume_text)
   

    chunks = []
    current_section = "Unknown"

    for i in range(len(sections)):
        text_content = sections[i].strip()

        if text_content in section_headers:
            current_section = text_content  # Assign proper section name
        elif text_content:
            try:
                chunk = ResumeChunk(section=current_section, text=text_content, chunk_id=f"chunk-{i//2}")
                chunks.append(chunk)
            except ValidationError as e:
                print(f"Skipping invalid chunk: {e}")

    return chunks

# Example usage
pdf_path = "D:\GenAI\code\data\Gowtham Mani Resume.pdf"  # Replace with your actual PDF resume file
resume_text = extract_text_from_pdf(pdf_path)
chunks = chunk_resume_with_labels(resume_text)



re.compile('(?i)^\\s*(SUMMARY|EXPERIENCE|EDUCATION|SKILLS & TOOLS|CERTIFICATIONS|PROJECTS)\\s*$', re.IGNORECASE|re.MULTILINE)
------------------------


In [57]:
chunks

[ResumeChunk(section='Unknown', text='Gowtham Mani\n7624773171 - gowthammani024@gmail.com - linkedin.com/in/gowthammani024/ - Dashboards - GitHub - Portfolio', chunk_id='chunk-0'),
 ResumeChunk(section='EXPERIENCE', text='Data Analytics Internship Oct 2024 - present\nRide for Change Ride for Hope Foundation, Chicago, USA\n• Supported data-driven decision-making through comprehensive data collection, cleaning, and organization, ensuring high-\nquality data for analysis.\n• Contributed to data visualization projects in Oracle NetSuite to deliver actionable insights for team initiatives & strategic\nplanning.\n• Assisted in diverse research tasks, collaborating with industry professionals to analyze trends and inform project strategies.\nData Analytics Apprenticeship Jan 2024 - Apr 2024\nKPMG Health Care, Orlando, USA\n• Developed predictive model for healthcare needs of four diseases for the state of Florida over future years for long-term planning\n& resource allocation.\n• Extracted, t

In [58]:
new_chunks=[]
for i in chunks:
    new_chunks.append(i.text)
new_chunks

['Gowtham Mani\n7624773171 - gowthammani024@gmail.com - linkedin.com/in/gowthammani024/ - Dashboards - GitHub - Portfolio',
 'Data Analytics Internship Oct 2024 - present\nRide for Change Ride for Hope Foundation, Chicago, USA\n• Supported data-driven decision-making through comprehensive data collection, cleaning, and organization, ensuring high-\nquality data for analysis.\n• Contributed to data visualization projects in Oracle NetSuite to deliver actionable insights for team initiatives & strategic\nplanning.\n• Assisted in diverse research tasks, collaborating with industry professionals to analyze trends and inform project strategies.\nData Analytics Apprenticeship Jan 2024 - Apr 2024\nKPMG Health Care, Orlando, USA\n• Developed predictive model for healthcare needs of four diseases for the state of Florida over future years for long-term planning\n& resource allocation.\n• Extracted, transferred & loaded 1M+ records from public databases. Used Python to cleanse the data into x da

**2)Converting the text into Embeddings**

In [59]:
#Embedding
from langchain.embeddings import HuggingFaceEmbeddings
# Initialize HuggingFace model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Embed the chunks
embeddings = embedding_model.embed_documents(new_chunks)


**3) Create a dictionary kind of format to load into the vector DB**

In [60]:
Final_emb=[]
for i in range(len(new_chunks)):
    d={}
    d['id']=str(i)
    d['values']=embeddings[i]
    l={}
    l[chunks[i].section]=chunks[i].text

    d['metadata']=l

    Final_emb.append(d)

In [61]:
for i in Final_emb:
    print(i)

{'id': '0', 'values': [-0.0714367926120758, -0.010662410408258438, -0.1026274710893631, -0.038501910865306854, 0.04242168739438057, 0.01465518493205309, -0.002865289803594351, 0.0436554029583931, -0.0054221078753471375, 0.0262614618986845, 0.02643761783838272, 0.015559324063360691, 0.0072844563983380795, -0.007236001547425985, -0.00415411964058876, 0.017388764768838882, 0.006884747184813023, -0.047627247869968414, 0.014046420343220234, -0.020083218812942505, -0.0716002881526947, -0.02484220638871193, 0.0005048296297900379, -0.040002256631851196, 0.03532234951853752, -0.02216012217104435, -0.08344775438308716, 0.01985061913728714, 0.016929615288972855, 0.061999205499887466, 0.03367646783590317, 0.01579590141773224, -0.0756390392780304, 0.007344549987465143, -0.061000142246484756, 0.04870898276567459, -0.007995683699846268, -0.0007653972133994102, 0.002695319941267371, 0.0234870333224535, -0.011113854125142097, 0.017356226220726967, -0.04288880154490471, -0.06800049543380737, 0.106307633

**4) Connecting PineCone to this Kernal**

In [62]:
from pinecone import Pinecone
pcobj=Pinecone(api_key="****")
index=pcobj.Index('firstrag')

**5)Connecting Langchain and Pinecone**

In [63]:
from langchain_pinecone import PineconeVectorStore
vector_store = PineconeVectorStore(index=index, embedding=embedding_model, namespace="Gowtham3rd Resume")

In [64]:
index.upsert(Final_emb)

{'upserted_count': 6}

**6)initializing LLM using Groq Model**

In [65]:
GROQ_API_KEY="****"

In [66]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama3-8b-8192",
    temperature=0.1,
    max_retries=2,
    # other params...
)

**7)Converting the Query into Embeddings**

In [67]:
query='tell about the education'
query_emb = embedding_model.embed_query(query)

**8)Storing the matched/similarity based docs in one variable**

In [68]:
result1 = index.query(vector=query_emb, top_k=3, include_metadata=True)

In [69]:
result1

{'matches': [{'id': '4',
              'metadata': {'Education': 'Master of Science in Data Analytics – '
                                        'University of Central Florida '
                                        'Aug-2017'},
              'score': 0.0759052113,
              'values': []},
             {'id': '0',
              'metadata': {'Unknown': 'Harish Suresh\n'
                                      '+1 (678) 749 5466 | '
                                      'harishsuresh802@gmail.com'},
              'score': 0.0737897232,
              'values': []},
             {'id': '2',
              'metadata': {'Environment and Tools': 'Data Visualization and '
                                                    'Reporting Tools- Power '
                                                    'BI, Excel, DAX, Power BI '
                                                    'Service, Power BI '
                                                    'Dataflows,\n'
                         

In [70]:
rt=[]

for i in range(len(result1['matches'])):
    d=result1['matches'][i]['metadata']
    for key,value in d.items():
        rt.append(value)
rt


['Master of Science in Data Analytics – University of Central Florida Aug-2017',
 'Harish Suresh\n+1 (678) 749 5466 | harishsuresh802@gmail.com',
 'Data Visualization and Reporting Tools- Power BI, Excel, DAX, Power BI Service, Power BI Dataflows,\nTableau, Looker, Amazon QuickSight\nDatabase Management - SQL, SQL Server, Teradata, Snowflake, MySQL, PostgreSQL, Database Query\nOptimization, Indexing, Data Modeling\nETL Processes and Data Integration - ETL, SQL Server, Snowflake, Teradata, Data Pipelines, Data\nTransformation, Data Warehousing, Apache Airflow\nCloud and Data Migration Technologies- WS S3, Azure, Cloud-Based Architectures, Cloud Data Storage, Cloud']

In [71]:
context="\n".join(["".join(text) for text in rt])
context

'Master of Science in Data Analytics – University of Central Florida Aug-2017\nHarish Suresh\n+1 (678) 749 5466 | harishsuresh802@gmail.com\nData Visualization and Reporting Tools- Power BI, Excel, DAX, Power BI Service, Power BI Dataflows,\nTableau, Looker, Amazon QuickSight\nDatabase Management - SQL, SQL Server, Teradata, Snowflake, MySQL, PostgreSQL, Database Query\nOptimization, Indexing, Data Modeling\nETL Processes and Data Integration - ETL, SQL Server, Snowflake, Teradata, Data Pipelines, Data\nTransformation, Data Warehousing, Apache Airflow\nCloud and Data Migration Technologies- WS S3, Azure, Cloud-Based Architectures, Cloud Data Storage, Cloud'

**9) Creating the prompt**

In [72]:
prompt = f"""
context:
{context}

Question:
{query}
"""

**10) Invoking the LLM**

In [73]:
llm.invoke([{"role": "ai","content": prompt}]).content

'Answer:\nI hold a Master of Science in Data Analytics from the University of Central Florida, which I completed in August 2017.'

In [74]:

while True:
    # Get user input
    user_query = input("Your query: ") #.strip()
    #print(user_query)
    if user_query.lower() == "exit":
        print("Exiting chatbot.")
        break

    # 7) Converting the query into embeddings
    query_emb = embedding_model.embed_query(user_query)
    #print("query embedding done")
    
    # 8) Retrieve the most similar resume chunks from Pinecone
    result1 = index.query(vector=query_emb, top_k=3, include_metadata=True)
    
    # Extract text from the metadata for context
    rt=[]

    for i in range(len(result1['matches'])):
        d=result1['matches'][i]['metadata']
        for key,value in d.items():
            rt.append(value)
    
    # Build the context for the prompt
    #context = "\n".join(retrieved_texts)
    context="\n".join(["".join(text) for text in rt])
    #print(context)

    
    # 9) Create the prompt combining context and query
    prompt = f"""
    Content:
    {context}

    Question:
    {user_query}
    """
    
    # 10) Invoke the LLM to get the response
    print(llm.invoke([{"role": "user", "content": prompt}]).content)

 





Hi! It's nice to meet you. I'm Gowtham Mani, a data analytics professional with a strong background in data analysis, visualization, and machine learning. I'm currently working as a Data Analytics Intern at Ride for Change Ride for Hope Foundation, where I'm supporting data-driven decision-making and contributing to data visualization projects.

I also have experience as a Data Analytics Apprenticeship at KPMG Health Care, where I developed predictive models for healthcare needs and constructed user interactive dashboards in Tableau. Prior to that, I worked as a Database Administrator at Cognizant Technology Solutions, where I formulated complex SQL queries and applied statistical methods to analyze large datasets.

I'm a recent graduate with a Master's degree in Data Analytics from the University of Central Florida and a Bachelor's degree in Robotics and Automation from PSG College of Technology. I'm excited to connect with you and explore potential opportunities in the field of data 